In [0]:
import  pyspark.sql.functions as F
from pyspark.sql.types import StringType, IntegerType, DateType,TimestampType,FloatType

catalog_name='ecommerce'

In [0]:
df_bronze=spark.table(f"{catalog_name}.bronze.brz_brands")
df_bronze.show()

##Remove spaces from Brand_name


In [0]:
df_silver=df_bronze.withColumn('brand_name',F.trim(F.col('brand_name')))

df_silver.show()

##Remove @ in brand_code which is extra

In [0]:

df_silver = df_silver.withColumn(
    "brand_id",
    F.regexp_replace(F.col("brand_id"), r'[^A-Za-z0-9]', '')
)

df_silver.show()

##Find Distinct Category Code

In [0]:
df_silver.select("category_code").distinct().show()

In [0]:
anomalies={
    "GROCERY":"GRCY",
    "BOOKS":"BKS",
    "TOYS":"TOY"
}

df_silver=df_silver.replace(anomalies,subset=["category_code"])

df_silver.select("category_code").distinct().show()


##Load All Cleanimg of Brand Tabke in silver layer

In [0]:
df_silver.write.format("delta")\
    .mode("overwrite")\
    .option("mergeSchema", "true")\
    .saveAsTable(f"{catalog_name}.silver.slv_brands")